In [5]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

data = {
    'order_id': ['O1','O2','O3','O4','O5','O6','O7','O8','O9','O10','O11','O12'],
    'city': ['Mumbai','Delhi','Mumbai','Bangalore','Delhi','Mumbai','Delhi',
             'Bangalore','Mumbai','Delhi','Bangalore','Mumbai'],
    'category': ['Tech','Furniture','Tech','Office','Tech','Furniture','Office',
                 'Tech','Office','Furniture','Furniture','Tech'],
    'sales': [25000, 8000, 15000, 3000, 32000,np.nan , 5500, 18000, 4200, 9800, 7600, 21000],
    'profit': [5000, -500, 3200, 800, 8000, 1200, 600, 4000, -300, 1900, 1100, 5200],
    'order_date': pd.to_datetime(['2025-01-05','2025-01-08','2025-02-10','2025-02-12',
                                   '2025-02-15','2025-03-01','2025-03-05','2025-03-10',
                                   '2025-03-15','2025-04-01','2025-04-05','2025-04-10'])
}
df = pd.DataFrame(data)
df

,order_id,city,category,sales,profit,order_date
0,O1,Mumbai,Tech,25000.0,5000,2025-01-05
1,O2,Delhi,Furniture,8000.0,-500,2025-01-08
2,O3,Mumbai,Tech,15000.0,3200,2025-02-10
3,O4,Bangalore,Office,3000.0,800,2025-02-12
4,O5,Delhi,Tech,32000.0,8000,2025-02-15
5,O6,Mumbai,Furniture,NaN,1200,2025-03-01
6,O7,Delhi,Office,5500.0,600,2025-03-05
7,O8,Bangalore,Tech,18000.0,4000,2025-03-10
8,O9,Mumbai,Office,4200.0,-300,2025-03-15
9,O10,Delhi,Furniture,9800.0,1900,2025-04-01


In [6]:
result = df.groupby(['city', 'category'])['sales'].sum()
result

city       category 
Bangalore  Furniture     7600.0
           Office        3000.0
           Tech         18000.0
Delhi      Furniture    17800.0
           Office        5500.0
           Tech         32000.0
Mumbai     Furniture        0.0
           Office        4200.0
           Tech         61000.0
Name: sales, dtype: float64

In [3]:
# What we WANT to see — a cross-tab, cities as rows, categories as columns
pd.DataFrame({
    'Furniture': [0, 17800, 12000],
    'Office':    [3000, 5500, 4200],
    'Tech':      [18000, 32000, 61000]
}, index=['Bangalore', 'Delhi', 'Mumbai'])

,Furniture,Office,Tech
Bangalore,0,3000,18000
Delhi,17800,5500,32000
Mumbai,12000,4200,61000


### The Syntax

```python
pd.pivot_table(
    data,            # your DataFrame
    values='col',    # which column to aggregate
    index='col',      # becomes the ROW labels
    columns='col',    # becomes the COLUMN labels
    aggfunc='sum',    # how to combine values at each intersection
    fill_value=0      # what to put when there's no data for a combination
)
```

Think of it like setting up a table in three steps:
1. **Rows** — what do you want going down the left side? (`index`)
2. **Columns** — what do you want going across the top? (`columns`)
3. **Cells** — what number goes where rows and columns meet, and how is it calculated? (`values` + `aggfunc`)

groupby()

"Split the data into groups so I can perform any kind of analysis or transformation."

pivot_table()

"Summarize the data into a report with rows and columns."

In [7]:
pivot = pd.pivot_table(
    df,
    values='sales',
    index='city',
    columns='category',
    aggfunc='sum',
    fill_value=0
)
pivot

category,Furniture,Office,Tech
city,,,
Bangalore,7600.0,3000.0,18000.0
Delhi,17800.0,5500.0,32000.0
Mumbai,0.0,4200.0,61000.0


In [10]:
pivot_totals = pd.pivot_table(
    df, values='sales', index='city', columns='category',
    aggfunc='sum', fill_value=0,
    margins=False,  margins_name='Total Sales'
)
pivot_totals

category,Furniture,Office,Tech
city,,,
Bangalore,7600.0,3000.0,18000.0
Delhi,17800.0,5500.0,32000.0
Mumbai,0.0,4200.0,61000.0


In [14]:
# You can pivot multiple metrics at once
multi_value = pd.pivot_table(
    df, values=['sales', 'profit'], index='city', columns='category',
    aggfunc='sum', fill_value=0, margins = True
)
multi_value

profit                             sales                     \
category  Furniture  Office     Tech    All Furniture   Office      Tech   
city                                                                       
Bangalore    1100.0   800.0   4000.0   5900    7600.0   3000.0   18000.0   
Delhi        1400.0   600.0   8000.0  10000   17800.0   5500.0   32000.0   
Mumbai       1200.0  -300.0  13400.0  13100       0.0   4200.0   61000.0   
All          2500.0  1100.0  25400.0  29000   25400.0  12700.0  111000.0   

                     
category        All  
city                 
Bangalore   28600.0  
Delhi       55300.0  
Mumbai      65200.0  
All        149100.0

In [12]:
np.random.seed(42)
n = 400
regions = ['West', 'East', 'Central', 'South']
categories = ['Furniture', 'Office Supplies', 'Technology']

rows = []
start_date = pd.Timestamp('2023-01-01')
for i in range(n):
    region = np.random.choice(regions)
    category = np.random.choice(categories, p=[0.25, 0.45, 0.30])
    order_date = start_date + pd.Timedelta(days=int(np.random.randint(0, 730)))
    sales = round(np.random.gamma(2, 150) + 20, 2)
    profit = round(sales * np.random.normal(0.12, 0.15), 2)
    rows.append([f"ORD-{i+1:05d}", order_date, region, category, sales, profit])

superstore = pd.DataFrame(rows, columns=['Order_ID', 'Order_Date', 'Region', 'Category', 'Sales', 'Profit'])
print(superstore.shape)
superstore.head()

(400, 6)


,Order_ID,Order_Date,Region,Category,Sales,Profit
0,ORD-00001,2023-09-28,Central,Technology,417.57,145.50
1,ORD-00002,2024-04-03,Central,Furniture,328.02,89.08
2,ORD-00003,2023-12-10,East,Furniture,189.66,38.19
3,ORD-00004,2023-11-10,West,Office Supplies,28.62,7.51
4,ORD-00005,2024-04-20,Central,Office Supplies,315.52,1.52


Using the `superstore` DataFrame above, answer these on your own before scrolling further:

1. Build a pivot table showing **average discount** by `Region` and `Segment`.
2. Which `Sub_Category` has the highest total `Quantity` sold? (Hint: pivot with `index='Sub_Category'`, no `columns` needed — a pivot table can have just one dimension.)
3. Add a grand total row/column to your answer from Question 1.

Try it in the empty cell below before checking the solution.

In [ ]:
# try the solution here



In [18]:
# Business question: "Which region is most profitable in each category?"
pd.pivot_table(
    superstore,            # your DataFrame
    values='Profit',    # which column to aggregate
    index='Category',      # becomes the ROW labels
    columns='Region',    # becomes the COLUMN labels
    aggfunc='sum',    # how to combine values at each intersection
    fill_value=0,      # what to put when there's no data for a combination
    margins = True, margins_name = "Total"
)

Region,Central,East,South,West,Total
Category,,,,,
Furniture,1504.25,1056.58,1892.25,1256.14,5709.22
Office Supplies,1710.11,1031.75,2605.07,1272.68,6619.61
Technology,917.34,1529.93,571.89,1343.84,4363.00
Total,4131.70,3618.26,5069.21,3872.66,16691.83


In [19]:
multi_agg_pivot = pd.pivot_table(
    df,
    values='sales',
    index='city',
    columns='category',
    aggfunc=['sum', 'mean'],   # multiple aggregations on the SAME value column
    fill_value=0
)
multi_agg_pivot.round(0)

sum                       mean                 
category  Furniture  Office     Tech Furniture  Office     Tech
city                                                           
Bangalore    7600.0  3000.0  18000.0    7600.0  3000.0  18000.0
Delhi       17800.0  5500.0  32000.0    8900.0  5500.0  32000.0
Mumbai          0.0  4200.0  61000.0       0.0  4200.0  20333.0

In [20]:
pivot = pd.pivot_table(
    df,
    values="sales",
    index="city",
    columns="category",
    aggfunc="sum"
)

In [21]:
pivot

category,Furniture,Office,Tech
city,,,
Bangalore,7600.0,3000.0,18000.0
Delhi,17800.0,5500.0,32000.0
Mumbai,0.0,4200.0,61000.0


In [22]:
pivot.loc["Average"] = pivot.mean()

In [23]:
pivot

category,Furniture,Office,Tech
city,,,
Bangalore,7600.000000,3000.000000,18000.0
Delhi,17800.000000,5500.000000,32000.0
Mumbai,0.000000,4200.000000,61000.0
Average,8466.666667,4233.333333,37000.0


In [24]:
pivot["Average"] = pivot.mean(axis=1)

pivot

category,Furniture,Office,Tech,Average
city,,,,
Bangalore,7600.000000,3000.000000,18000.0,9533.333333
Delhi,17800.000000,5500.000000,32000.0,18433.333333
Mumbai,0.000000,4200.000000,61000.0,21733.333333
Average,8466.666667,4233.333333,37000.0,16566.666667


In [25]:
multi_value_pivot = pd.pivot_table(
    df,
    values=['sales', 'profit'],   # list instead of a single string
    index='city',
    columns='category',
    aggfunc='sum',
    fill_value=0
)
multi_value_pivot

profit                   sales                 
category  Furniture Office   Tech Furniture  Office     Tech
city                                                        
Bangalore      1100    800   4000    7600.0  3000.0  18000.0
Delhi          1400    600   8000   17800.0  5500.0  32000.0
Mumbai         1200   -300  13400       0.0  4200.0  61000.0

In [27]:
print("Before flattening:")
print(multi_value_pivot.columns.tolist())

Before flattening:
[('profit', 'Furniture'), ('profit', 'Office'), ('profit', 'Tech'), ('sales', 'Furniture'), ('sales', 'Office'), ('sales', 'Tech')]


In [33]:
flat_pivot = multi_value_pivot.copy()

for col in flat_pivot:
  print('_'.join(col))


profit_Furniture
profit_Office
profit_Tech
sales_Furniture
sales_Office
sales_Tech


In [28]:
flat_pivot.columns = ['_'.join(col).strip() for col in flat_pivot.columns.values]

flat_pivot

,profit_Furniture,profit_Office,profit_Tech,sales_Furniture,sales_Office,sales_Tech
city,,,,,,
Bangalore,1100,800,4000,7600.0,3000.0,18000.0
Delhi,1400,600,8000,17800.0,5500.0,32000.0
Mumbai,1200,-300,13400,0.0,4200.0,61000.0


In [26]:
# Recall: multi_value_pivot has MultiIndex columns like ('profit', 'Furniture')
print("Before flattening:")
print(multi_value_pivot.columns.tolist())

flat_pivot = multi_value_pivot.copy()
flat_pivot.columns = ['_'.join(col).strip() for col in flat_pivot.columns.values]

print("\nAfter flattening:")
print(flat_pivot.columns.tolist())
flat_pivot

Before flattening:
[('profit', 'Furniture'), ('profit', 'Office'), ('profit', 'Tech'), ('sales', 'Furniture'), ('sales', 'Office'), ('sales', 'Tech')]

After flattening:
['profit_Furniture', 'profit_Office', 'profit_Tech', 'sales_Furniture', 'sales_Office', 'sales_Tech']


,profit_Furniture,profit_Office,profit_Tech,sales_Furniture,sales_Office,sales_Tech
city,,,,,,
Bangalore,1100,800,4000,7600.0,3000.0,18000.0
Delhi,1400,600,8000,17800.0,5500.0,32000.0
Mumbai,1200,-300,13400,0.0,4200.0,61000.0


In [35]:
x = [1,2,3,4,5]

y = []

for value in x:
  y.append(value**2)

y

[1, 4, 9, 16, 25]

In [36]:
y = [value**2 for value in x]

y

[1, 4, 9, 16, 25]

## **Running SQL in Google Colab**

No installation step is needed — `sqlite3` ships with Python.

In [37]:
import sqlite3
import pandas as pd

# Step 1: Create a CONNECTION to a database.
# ':memory:' means "don't save to disk — build the database in RAM, just for this session"
conn = sqlite3.connect(':memory:')

print("Connection created:", conn)

Connection created: <sqlite3.Connection object at 0x7d44ff0527a0>


**What just happened?**
- `sqlite3.connect(':memory:')` created a brand-new, empty SQLite database that exists only in your computer's memory.
- Use `':memory:'` for learning and quick analysis — it disappears when your Colab session ends.

In [38]:
# Step 2: Get some data into the database.
# We'll build two small DataFrames first (Customers and Orders), then load them as SQL tables.

customers = pd.DataFrame({
    'customer_id': ['C1', 'C2', 'C3', 'C4', 'C5'],
    'name':        ['Alice', 'Bob', 'Charlie', 'Diana', 'Evan'],
    'city':        ['Mumbai', 'Delhi', 'Mumbai', 'Bangalore', 'Delhi'],
    'tier':        ['Gold', 'Silver', 'Gold', 'Silver', 'Gold']
})

orders = pd.DataFrame({
    'order_id':    ['O1', 'O2', 'O3', 'O4', 'O5', 'O6', 'O7'],
    'customer_id': ['C1', 'C2', 'C1', 'C3', 'C4', 'C6', 'C2'],   # Note: C6 has no matching customer!
    'product':     ['Laptop', 'Chair', 'Mouse', 'Desk', 'Phone', 'Tablet', 'Monitor'],
    'amount':      [55000, 8000, 1200, 15000, 32000, 22000, 18000],
    'order_date':  ['2025-01-10', '2025-01-15', '2025-02-01', '2025-02-10',
                     '2025-03-01', '2025-03-05', '2025-03-10']
})

# Step 3: Load each DataFrame into the database AS A TABLE.
# if_exists='replace' means: if a table with this name already exists, overwrite it.
customers.to_sql('customers', conn, index=False, if_exists='replace')
orders.to_sql('orders', conn, index=False, if_exists='replace')

print("Tables created inside the database.")

Tables created inside the database.


In [39]:
customers

,customer_id,name,city,tier
0,C1,Alice,Mumbai,Gold
1,C2,Bob,Delhi,Silver
2,C3,Charlie,Mumbai,Gold
3,C4,Diana,Bangalore,Silver
4,C5,Evan,Delhi,Gold


In [40]:
orders

,order_id,customer_id,product,amount,order_date
0,O1,C1,Laptop,55000,2025-01-10
1,O2,C2,Chair,8000,2025-01-15
2,O3,C1,Mouse,1200,2025-02-01
3,O4,C3,Desk,15000,2025-02-10
4,O5,C4,Phone,32000,2025-03-01
5,O6,C6,Tablet,22000,2025-03-05
6,O7,C2,Monitor,18000,2025-03-10


In [41]:
# Step 4: Write a helper so we don't repeat ourselves on every query
def sql(query):
    """Run a SQL query against our database and return the result as a Pandas DataFrame."""
    return pd.read_sql_query(query, conn)

In [42]:
sql("SELECT * FROM customers")

,customer_id,name,city,tier
0,C1,Alice,Mumbai,Gold
1,C2,Bob,Delhi,Silver
2,C3,Charlie,Mumbai,Gold
3,C4,Diana,Bangalore,Silver
4,C5,Evan,Delhi,Gold


**Explaining every piece of `pd.read_sql_query(query, conn)`:**
- `query` — the SQL text you want to run, as a Python string
- `conn` — which database connection to run it against
- The return value is automatically a Pandas DataFrame — this is the magic that makes SQL and Pandas feel like one connected workflow rather than two separate worlds

This `sql()` helper function is exactly what you'll use for every query for the rest of this notebook.

In [43]:
sql("SELECT * FROM orders")

,order_id,customer_id,product,amount,order_date
0,O1,C1,Laptop,55000,2025-01-10
1,O2,C2,Chair,8000,2025-01-15
2,O3,C1,Mouse,1200,2025-02-01
3,O4,C3,Desk,15000,2025-02-10
4,O5,C4,Phone,32000,2025-03-01
5,O6,C6,Tablet,22000,2025-03-05
6,O7,C2,Monitor,18000,2025-03-10


The business question: Which customers have spent more than ₹20,000 in total?

In [48]:
sql('''
SELECT customer_id, SUM(amount) AS total_spent
FROM orders
WHERE SUM(amount) > 20000
GROUP BY customer_id
''')

DatabaseError: Execution failed on sql '
SELECT customer_id, SUM(amount) AS total_spent
FROM orders
WHERE SUM(amount) > 20000
GROUP BY customer_id
': misuse of aggregate: SUM()

**That works.** `HAVING` filters **groups**, and it runs **after** `GROUP BY` has already collapsed rows into groups and computed the aggregates. By the time `HAVING` runs, `SUM(amount)` already exists as a real number per group — so comparing it to 20000 makes sense.

### SQL's Real Execution Order

This is one of the most important mental models in all of SQL. The order you *write* a query is **not** the order the database *executes* it in:

```
You WRITE:                    Database EXECUTES:

SELECT   ...                  1. FROM        (get the raw table)
FROM     ...                       ↓
WHERE    ...                  2. WHERE       (filter individual rows)
GROUP BY ...                       ↓
HAVING   ...                  3. GROUP BY    (collapse rows into groups)
ORDER BY ...                       ↓
                               4. HAVING      (filter groups)
                                    ↓
                               5. SELECT      (pick/compute final columns)
                                    ↓
                               6. ORDER BY    (sort the final result)
                                    ↓
                               7. LIMIT       (cut down to N rows)
```

In [49]:
# Side-by-side comparison: WHERE filters rows, HAVING filters groups

print("WHERE: only orders above 20000 (row-level filter, no grouping)")
print(sql("SELECT * FROM orders WHERE amount > 20000"))



WHERE: only orders above 20000 (row-level filter, no grouping)
  order_id customer_id product  amount  order_date
0       O1          C1  Laptop   55000  2025-01-10
1       O5          C4   Phone   32000  2025-03-01
2       O6          C6  Tablet   22000  2025-03-05


In [50]:
print("\nHAVING: only customers whose TOTAL exceeds 20000 (group-level filter)")
print(sql('''
    SELECT customer_id, SUM(amount) AS total_spent
    FROM orders GROUP BY customer_id
    HAVING SUM(amount) > 20000
'''))


HAVING: only customers whose TOTAL exceeds 20000 (group-level filter)
  customer_id  total_spent
0          C1        56200
1          C2        26000
2          C4        32000
3          C6        22000


In [51]:
# "Among Tech-category-style high-value orders (above 5000), which customers
#  still have a total above 20000?"

sql('''
    SELECT customer_id
    FROM orders
    WHERE amount > 5000
    GROUP BY customer_id
    HAVING SUM(amount) > 20000
''')

,customer_id
0,C1
1,C2
2,C4
3,C6
